# Optional Colab MLP baseline

Original 70 → 64 ReLU → 4 logit model, trained on generated shots; no existing QEC decoder implementation. No competition submission is made. Use a free CPU runtime initially. GPU is optional; no paid Compute Engine connection is required. Training/validation sets use separate Stim seeds. The base repo does not require PyTorch.

Review the embedded code before running. This notebook is self-contained: it writes the original NumPy inference and PyTorch training scripts into the Colab runtime and downloads only the organizer’s two pinned circuit files. Exported inference requires NumPy and Stim, not PyTorch. Raw private transcript data are never used.


In [ ]:
%pip install -q numpy==2.2.6 stim==1.16.0
# Colab normally includes PyTorch. If unavailable, use its supported runtime installation.
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import torch
print("PyTorch", torch.__version__, "CUDA available", torch.cuda.is_available())


In [ ]:
from pathlib import Path
import json, urllib.request
Path('data/kaggle').mkdir(parents=True, exist_ok=True)
files = json.loads('{"decoder.py": "\\"\\"\\"Portable NumPy inference for the team\'s two-layer ReLU network.\\"\\"\\"\\n\\nfrom functools import lru_cache\\nfrom hashlib import sha256\\nfrom pathlib import Path\\nimport numpy as np\\nimport stim\\n\\n\\ndef fingerprint(circuit):\\n    return sha256(str(circuit).encode()).hexdigest()\\n\\n\\nclass MLP:\\n    def __init__(self, w0, b0, w1, b1):\\n        self.w0, self.b0, self.w1, self.b1 = (\\n            np.asarray(x, dtype=np.float32) for x in (w0, b0, w1, b1)\\n        )\\n        if (\\n            self.w0.ndim != 2\\n            or self.w1.ndim != 2\\n            or self.b0.shape != (self.w0.shape[1],)\\n            or self.w1.shape[0] != self.w0.shape[1]\\n            or self.b1.shape != (self.w1.shape[1],)\\n        ):\\n            raise ValueError(\\"Invalid network shapes\\")\\n        if not all(np.isfinite(x).all() for x in (self.w0, self.b0, self.w1, self.b1)):\\n            raise ValueError(\\"Nonfinite network weights\\")\\n\\n    @classmethod\\n    def load(cls, path, circuit_hash):\\n        with np.load(path, allow_pickle=False) as f:\\n            if str(f[\\"circuit_hash\\"]) != circuit_hash:\\n                raise ValueError(\\"Circuit fingerprint mismatch\\")\\n            return cls(f[\\"w0\\"], f[\\"b0\\"], f[\\"w1\\"], f[\\"b1\\"])\\n\\n    def logits(self, events):\\n        a = np.asarray(events)\\n        if a.ndim != 2 or a.shape[1] != len(self.w0) or not np.isin(a, (0, 1)).all():\\n            raise ValueError(\\"Invalid detector array\\")\\n        return (\\n            np.maximum(a.astype(np.float32) @ self.w0 + self.b0, 0) @ self.w1 + self.b1\\n        )\\n\\n    def predict(self, events):\\n        return (self.logits(events) > 0).astype(np.uint8)\\n\\n\\n@lru_cache(maxsize=2)\\ndef _model(key):\\n    return MLP.load(Path(__file__).parent / \\"models\\" / f\\"{key}.npz\\", key)\\n\\n\\ndef decode(circuit: stim.Circuit, detector_events: np.ndarray) -> np.ndarray:\\n    return _model(fingerprint(circuit)).predict(detector_events)\\n", "train.py": "\\"\\"\\"Optional PyTorch training; inference exports use NumPy and Stim only.\\n\\nRun in Colab\'s existing PyTorch environment or a separate training environment.\\nAll train/validation shots are independently generated from the supplied circuits.\\n\\"\\"\\"\\n\\nimport argparse\\nfrom hashlib import sha256\\nimport importlib.metadata as metadata\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport platform\\nimport numpy as np\\nimport stim\\nfrom decoder import MLP, fingerprint\\n\\n\\ndef train(circuit, out, shots=50000, epochs=5, hidden=64, seed=41, batch_size=512):\\n    # Set before PyTorch/CUDA initialization for deterministic matrix products.\\n    os.environ.setdefault(\\"CUBLAS_WORKSPACE_CONFIG\\", \\":4096:8\\")\\n    import torch\\n\\n    torch.manual_seed(seed)\\n    if torch.cuda.is_available():\\n        torch.cuda.manual_seed_all(seed)\\n    torch.use_deterministic_algorithms(True)\\n    device = \\"cuda\\" if torch.cuda.is_available() else \\"cpu\\"\\n    events, labels = circuit.compile_detector_sampler(seed=seed).sample(\\n        shots, separate_observables=True\\n    )\\n    validation, truth = circuit.compile_detector_sampler(seed=seed + 1000000).sample(\\n        10000, separate_observables=True\\n    )\\n    x = torch.tensor(events, dtype=torch.float32, device=device)\\n    y = torch.tensor(labels, dtype=torch.float32, device=device)\\n    model = torch.nn.Sequential(\\n        torch.nn.Linear(circuit.num_detectors, hidden),\\n        torch.nn.ReLU(),\\n        torch.nn.Linear(hidden, circuit.num_observables),\\n    ).to(device)\\n    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)\\n    loss_fn = torch.nn.BCEWithLogitsLoss()\\n    generator = torch.Generator(device=device).manual_seed(seed)\\n    for epoch in range(epochs):\\n        order = torch.randperm(shots, generator=generator, device=device)\\n        for start in range(0, shots, batch_size):\\n            indices = order[start : start + batch_size]\\n            optimizer.zero_grad()\\n            loss = loss_fn(model(x[indices]), y[indices])\\n            loss.backward()\\n            optimizer.step()\\n        print(f\\"epoch {epoch + 1}: last minibatch BCE {loss.detach().item():.5f}\\", flush=True)\\n    key = fingerprint(circuit)\\n    out.mkdir(parents=True, exist_ok=True)\\n    path = out / f\\"{key}.npz\\"\\n    pending = out / f\\"{key}.pending.npz\\"\\n    if path.exists() or path.with_suffix(\\".json\\").exists() or pending.exists():\\n        raise FileExistsError(path)\\n    params = {\\n        name: value.detach().cpu().numpy() for name, value in model.state_dict().items()\\n    }\\n    with pending.open(\\"xb\\") as handle:\\n        np.savez_compressed(\\n            handle,\\n            w0=params[\\"0.weight\\"].T,\\n            b0=params[\\"0.bias\\"],\\n            w1=params[\\"2.weight\\"].T,\\n            b1=params[\\"2.bias\\"],\\n            circuit_hash=key,\\n        )\\n    exported = MLP.load(pending, key)\\n    with torch.no_grad():\\n        reference = (\\n            model(torch.tensor(validation[:1000], dtype=torch.float32, device=device))\\n            .cpu()\\n            .numpy()\\n        )\\n    np.testing.assert_allclose(\\n        exported.logits(validation[:1000]), reference, rtol=1e-4, atol=1e-5\\n    )\\n    # All reported predictions use the hand-in NumPy path, not the training framework.\\n    predictions = exported.predict(validation)\\n    record = {\\n        \\"circuit_hash\\": key,\\n        \\"seed\\": seed,\\n        \\"validation_seed\\": seed + 1000000,\\n        \\"shots\\": shots,\\n        \\"validation_shots\\": len(truth),\\n        \\"epochs\\": epochs,\\n        \\"hidden\\": hidden,\\n        \\"batch_size\\": batch_size,\\n        \\"device\\": device,\\n        \\"python\\": platform.python_version(),\\n        \\"packages\\": {p: metadata.version(p) for p in [\\"torch\\", \\"numpy\\", \\"stim\\"]},\\n        \\"validation_ler\\": float(np.mean(predictions != truth)),\\n        \\"zero_ler\\": float(truth.mean()),\\n        \\"weights_sha256\\": sha256(pending.read_bytes()).hexdigest(),\\n        \\"training_source_sha256\\": sha256(Path(__file__).read_bytes()).hexdigest(),\\n    }\\n    # Publish only after parity passes. Exclusive sidecar/link creation prevents\\n    # replacing a prior selected artifact; interrupted pending files remain visible.\\n    with path.with_suffix(\\".json\\").open(\\"x\\") as handle:\\n        handle.write(json.dumps(record, indent=2) + \\"\\\\n\\")\\n    os.link(pending, path)\\n    pending.unlink()\\n    print(json.dumps(record, indent=2), flush=True)\\n    return record\\n\\n\\nif __name__ == \\"__main__\\":\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\\"--circuits\\", type=Path, default=Path(\\"data/kaggle\\"))\\n    parser.add_argument(\\"--out\\", type=Path, default=Path(__file__).parent / \\"models\\")\\n    parser.add_argument(\\"--shots\\", type=int, default=50000)\\n    parser.add_argument(\\"--epochs\\", type=int, default=5)\\n    parser.add_argument(\\"--hidden\\", type=int, default=64)\\n    parser.add_argument(\\"--seed\\", type=int, default=41)\\n    args = parser.parse_args()\\n    if min(args.shots, args.epochs, args.hidden) < 1:\\n        parser.error(\\"shots,epochs,hidden must be positive\\")\\n    for i, b in enumerate(\\"XZ\\"):\\n        train(\\n            stim.Circuit.from_file(args.circuits / f\\"{b}_circuit.stim\\"),\\n            args.out,\\n            args.shots,\\n            args.epochs,\\n            args.hidden,\\n            args.seed + i,\\n        )\\n", "requirements.txt": "numpy==2.2.6\\nstim==1.16.0\\n", "package.py": "\\"\\"\\"Package only complete, hash-verified X/Z exports; never overwrite an archive.\\"\\"\\"\\n\\nfrom hashlib import sha256\\nimport json\\nfrom pathlib import Path\\nimport zipfile\\nimport stim\\n\\n\\ndef package(folder: Path, circuits: Path, output: Path):\\n    paths = [folder / name for name in (\\"decoder.py\\", \\"requirements.txt\\", \\"train.py\\")]\\n    for basis in \\"XZ\\":\\n        circuit = stim.Circuit.from_file(circuits / f\\"{basis}_circuit.stim\\")\\n        key = sha256(str(circuit).encode()).hexdigest()\\n        weights = folder / \\"models\\" / f\\"{key}.npz\\"\\n        record = weights.with_suffix(\\".json\\")\\n        data = json.loads(record.read_text())\\n        if (\\n            data[\\"circuit_hash\\"] != key\\n            or data[\\"weights_sha256\\"] != sha256(weights.read_bytes()).hexdigest()\\n        ):\\n            raise ValueError(f\\"Unverified {basis} model export\\")\\n        paths.extend([weights, record])\\n    # Read every input before creating anything; incomplete candidates leave no ZIP.\\n    contents = {str(path.relative_to(folder)): path.read_bytes() for path in paths}\\n    with zipfile.ZipFile(output, \\"x\\", compression=zipfile.ZIP_DEFLATED) as archive:\\n        for name, data in contents.items():\\n            archive.writestr(name, data)\\n    return output\\n"}')
for name, contents in files.items():
    Path(name).write_text(contents)
for basis in 'XZ':
    url = f'https://raw.githubusercontent.com/JuliusRye/QEC_Decoder_Hakathon_2026/2614389/data/train/{basis}_circuit.stim'
    urllib.request.urlretrieve(url, f'data/kaggle/{basis}_circuit.stim')


## Small smoke run first

This checks training/export mechanics with 2,000 training shots per basis and one epoch. It is not a tuned accuracy result. A separate 10,000-shot generated validation set is evaluated through the exported NumPy inference. Record the printed versions, seeds and weight hashes.


In [ ]:
!python train.py --shots 2000 --epochs 1 --out smoke_models


## Larger exploratory run

Only run after inspecting the smoke results and agreeing a compute budget. Start with 50,000 training shots per basis and five epochs. Each output directory must be new to preserve prior weights. Compare methods on identical independently generated holdout samples; the script’s validation score alone does not establish an advantage over BP.


In [ ]:
# Uncomment when ready:
# !python train.py --shots 50000 --epochs 5 --out models


## Download the hand-in files

After the larger run, bundle `decoder.py`, exact inference requirements, models, and the training source plus its recorded environment. The archive below is prepared only when `models/` exists. Never hand in the Colab virtual environment. Test from the extracted folder in a fresh Python environment.


In [ ]:
from pathlib import Path
from package import package
if Path('models').exists():
    package(Path('.'), Path('data/kaggle'), Path('handin_mlp.zip'))
    from google.colab import files
    files.download('handin_mlp.zip')
else:
    print('No final models yet; smoke weights are not packaged as a final candidate.')
